# VDA Report Explorer
Inspect the raw `theta_hat` matrices saved in `data/reports/` and summarize results.

In [ ]:
import numpy as np
import json
from pathlib import Path

SUBSET = "Hand-Crafted"
REPORT_DIR = Path("data/reports") / SUBSET

manifest = json.load(open(REPORT_DIR / "manifest.json"))
report_files = sorted(REPORT_DIR.glob("*.npz"), key=lambda f: int(f.stem))

print(f"Model      : {manifest['model']}")
print(f"Temperatures: {manifest['temperatures']}")
print(f"K          : {manifest['K']}")
print(f"Saved traces: {len(report_files)}")

## 1. Raw data — pick a trace

In [ ]:
TRACE_ID = 1   # change this to look at different traces

data = np.load(REPORT_DIR / f"{TRACE_ID}.npz", allow_pickle=True)
theta = data["theta_hat"]          # shape (K, T)
model_ids = list(data["model_ids"])
mistake_agent = str(data["mistake_agent"])
mistake_step  = int(data["mistake_step"])
K, T = theta.shape

print(f"Trace {TRACE_ID}")
print(f"  T (steps)     : {T}")
print(f"  mistake_agent : {mistake_agent}")
print(f"  mistake_step  : {mistake_step}  <- ground truth")
print(f"  model_ids     : {model_ids}")
print()
print("theta_hat raw (K x T):")
print("  rows = discriminators, columns = steps")
print(np.round(theta, 3))

In [ ]:
# Print as a readable table: step | D0 | D1 | D2 | mean | ground_truth?
print(f"{'step':>5} | {'D0':>6} | {'D1':>6} | {'D2':>6} | {'mean':>6} | GT")
print("-" * 48)
for t in range(T):
    vals = theta[:, t]
    gt = " <-- MISTAKE" if t == mistake_step else ""
    print(f"{t:>5} | {vals[0]:>6.3f} | {vals[1]:>6.3f} | {vals[2]:>6.3f} | {vals.mean():>6.3f} |{gt}")

## 2. Heatmap of theta_hat

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(max(10, T * 0.2), 2.5))
im = ax.imshow(theta, aspect="auto", vmin=0, vmax=1, cmap="RdYlGn_r")
plt.colorbar(im, ax=ax, label="P(root-cause error)")
ax.set_yticks(range(K))
ax.set_yticklabels([f"D{k} (T={manifest['temperatures'][k]})" for k in range(K)])
ax.set_xlabel("Step")
ax.set_title(f"Trace {TRACE_ID} | mistake: step={mistake_step} agent={mistake_agent}")
ax.axvline(mistake_step, color="blue", linewidth=2, linestyle="--", label=f"ground truth step {mistake_step}")
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

## 3. Mean probability per step

In [ ]:
mean_theta = theta.mean(axis=0)
predicted_step = int(np.argmax(mean_theta))

fig, ax = plt.subplots(figsize=(max(10, T * 0.2), 3))
colors = ["red" if t == mistake_step else "steelblue" for t in range(T)]
ax.bar(range(T), mean_theta, color=colors, alpha=0.8)
ax.axhline(0.5, color="gray", linestyle="--", linewidth=1, label="threshold=0.5")
ax.axvline(mistake_step, color="red", linewidth=2, linestyle="--", label=f"ground truth step {mistake_step}")
ax.axvline(predicted_step, color="orange", linewidth=2, linestyle=":", label=f"predicted step {predicted_step}")
ax.set_xlabel("Step")
ax.set_ylabel("Mean P(root-cause)")
ax.set_title(f"Trace {TRACE_ID} — mean theta per step")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print(f"Predicted step : {predicted_step}")
print(f"Ground truth   : {mistake_step}")
print(f"Correct        : {predicted_step == mistake_step}")

## 4. Summary across all saved traces

In [ ]:
rows = []
for f in report_files:
    d = np.load(f, allow_pickle=True)
    th = d["theta_hat"]
    ms = int(d["mistake_step"])
    ma = str(d["mistake_agent"])
    mean_per_step = th.mean(axis=0)
    pred_step = int(np.argmax(mean_per_step))
    rows.append({
        "trace_id"       : int(f.stem),
        "T"              : th.shape[1],
        "mistake_agent"  : ma,
        "mistake_step"   : ms,
        "predicted_step" : pred_step,
        "step_correct"   : pred_step == ms,
        "mean_theta"     : round(float(mean_per_step.mean()), 4),
        "std_theta"      : round(float(mean_per_step.std()), 4),
        "min_theta"      : round(float(mean_per_step.min()), 4),
        "max_theta"      : round(float(mean_per_step.max()), 4),
    })

import pandas as pd
df = pd.DataFrame(rows)

acc = df["step_correct"].mean()
print(f"Acc_step (simple mean argmax): {acc:.2%}  ({df['step_correct'].sum()}/{len(df)} traces)")
print()
df

In [ ]:
# Distribution of mean theta across all traces
fig, axes = plt.subplots(1, 2, figsize=(12, 3))

axes[0].hist(df["mean_theta"], bins=20, color="steelblue", edgecolor="white")
axes[0].set_xlabel("Mean theta (per trace)")
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of mean theta across traces")

axes[1].hist(df["std_theta"], bins=20, color="coral", edgecolor="white")
axes[1].set_xlabel("Std theta (per trace)")
axes[1].set_ylabel("Count")
axes[1].set_title("Distribution of std theta (signal spread)")

plt.tight_layout()
plt.show()

## 5. All traces — mean theta heatmap (traces × steps)

In [ ]:
# Pad all traces to the same length for a 2D heatmap (traces x steps)
max_T = max(int(np.load(f, allow_pickle=True)["theta_hat"].shape[1]) for f in report_files)
n = len(report_files)

grid = np.full((n, max_T), np.nan)
mistake_steps = []
for i, f in enumerate(report_files):
    d = np.load(f, allow_pickle=True)
    th = d["theta_hat"].mean(axis=0)
    grid[i, :len(th)] = th
    mistake_steps.append(int(d["mistake_step"]))

fig, ax = plt.subplots(figsize=(min(max_T * 0.25, 24), n * 0.5 + 1))
im = ax.imshow(grid, aspect="auto", vmin=0, vmax=1, cmap="RdYlGn_r", interpolation="nearest")
plt.colorbar(im, ax=ax, label="Mean P(root-cause)")

# Mark each trace's ground-truth mistake step
for i, ms in enumerate(mistake_steps):
    ax.plot(ms, i, marker="x", color="blue", markersize=8, markeredgewidth=2)

ax.set_yticks(range(n))
ax.set_yticklabels([f"trace {int(f.stem)}" for f in report_files], fontsize=8)
ax.set_xlabel("Step")
ax.set_title("All traces — mean theta per step  (blue X = ground truth mistake step)")
plt.tight_layout()
plt.show()

## 6. Where does the ground-truth step rank? (rank of mistake_step by mean theta)

In [ ]:
# For each trace: rank the mistake_step among all steps (1 = highest mean theta)
rank_rows = []
for f in report_files:
    d = np.load(f, allow_pickle=True)
    th = d["theta_hat"].mean(axis=0)
    ms = int(d["mistake_step"])
    # rank 1 = best
    rank = int(np.sum(th >= th[ms]))   # how many steps score >= mistake step
    rank_rows.append({
        "trace_id": int(f.stem),
        "T": len(th),
        "mistake_step": ms,
        "mistake_step_mean_theta": round(float(th[ms]), 4),
        "rank_of_mistake_step": rank,
        "rank_pct": round(rank / len(th) * 100, 1),   # lower = better (1% = top)
    })

df_rank = pd.DataFrame(rank_rows)
print("Rank 1 = model scored mistake step highest of all steps")
print(f"Traces where mistake_step is ranked #1 : {(df_rank['rank_of_mistake_step']==1).sum()}/{len(df_rank)}")
print(f"Traces where mistake_step is in top-3  : {(df_rank['rank_of_mistake_step']<=3).sum()}/{len(df_rank)}")
print(f"Mean rank of mistake_step              : {df_rank['rank_of_mistake_step'].mean():.1f}")
print()
df_rank

## 7. Signal quality: does the mistake step stand out, or is everything flat?

In [ ]:
# For each trace: how much does the mistake step score above the trace mean?
# A high "lift" means the signal is sharp; near-zero means everything looks the same.
signal_rows = []
for f in report_files:
    d = np.load(f, allow_pickle=True)
    th = d["theta_hat"].mean(axis=0)
    ms = int(d["mistake_step"])
    lift = float(th[ms] - th.mean())
    signal_rows.append({
        "trace_id": int(f.stem),
        "T": len(th),
        "mistake_step_theta": round(float(th[ms]), 4),
        "trace_mean_theta": round(float(th.mean()), 4),
        "lift": round(lift, 4),        # positive = mistake step scores above average
        "std_theta": round(float(th.std()), 4),
    })

df_sig = pd.DataFrame(signal_rows)
print("'lift' = mistake_step theta - trace mean theta")
print("  positive = model scores mistake step higher than average (good)")
print("  near 0   = everything looks the same (hard case)")
print()

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
colors = ["green" if v > 0 else "red" for v in df_sig["lift"]]
axes[0].bar(df_sig["trace_id"], df_sig["lift"], color=colors, alpha=0.8)
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_xlabel("Trace ID")
axes[0].set_ylabel("Lift")
axes[0].set_title("Lift of mistake step above trace mean theta")

axes[1].scatter(df_sig["std_theta"], df_sig["lift"], alpha=0.7, color="steelblue")
for _, row in df_sig.iterrows():
    axes[1].annotate(str(int(row["trace_id"])), (row["std_theta"], row["lift"]), fontsize=7)
axes[1].axhline(0, color="gray", linestyle="--")
axes[1].set_xlabel("Std of theta (spread across steps)")
axes[1].set_ylabel("Lift")
axes[1].set_title("Lift vs spread  (top-right = easy traces)")

plt.tight_layout()
plt.show()
df_sig